In [0]:
#display the data from the table departments
display(spark.read.table("plworkforce_catalog.001_bronze.departments"))

from pyspark.sql import functions as F

# Use silver schema
spark.sql("USE plworkforce_catalog.002_silver")

# Read from Bronze
df = spark.table("plworkforce_catalog.001_bronze.departments")

# Transform
df_clean = (df

    # Clean text columns
    .withColumn("department_code", F.upper(F.trim("department_code")))
    .withColumn("department_name", F.initcap(F.trim("department_name")))
    .withColumn("cost_center", F.upper(F.trim("cost_center")))

    # Data quality
    .dropDuplicates(["department_id"])
    .filter(F.col("department_id").isNotNull())
)

# Write to Silver
(df_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("departments")
)

print("Silver departments created")

# to display the data from silver departments
display(spark.read.table("plworkforce_catalog.002_silver.departments"))